In [0]:
source_path='/Volumes/myorg_dev/raw/raw_data/fraud_watch_list_files/'

In [0]:
input_stream=(spark.readStream
              .format("cloudFiles")
              .option("cloudFiles.format", "json")
              .option("cloudFiles.schemaLocation", "/Volumes/myorg_dev/raw/raw_data/schema_location")
              .option("cloudFiles.inferColumnTypes","true")
              .load(source_path)
)

In [0]:
from pyspark.sql import functions as F
tranformed_df=input_stream.select(
"*",
F.col("_metadata.file_path").alias("file_path"),
F.current_timestamp().alias("ingestion_timestamp")
)

In [0]:
streaming_query=(tranformed_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation", "/Volumes/myorg_dev/raw/raw_data/checkpoint_for_autoloader/")
.trigger(availableNow=True)
.toTable("myorg_dev.raw.src_fraud_autoloader_test")
)

In [0]:
%sql
select * from myorg_dev.raw.src_fraud_autoloader_test